In [88]:
import cv2
import pickle
import os
import matplotlib.pyplot as plt
from PIL import Image
import cvzone
import numpy as np

In [89]:
img_1='frame.jpg'

# Always starting with new Positions 
if os.path.exists('Pos'):
    os.remove('Pos')

posList = []

drawing = False
ix, iy = -1, -1
newPos = None
delete_mode = False
deleteList = [] 


The bellow function handles mouse events for drawing and deleting slot rectangles on an image using OpenCV.

### Function: `mouseClick(event, x, y, flags, param)`

#### Global Variables Used:
- `ix, iy`: Initial coordinates when mouse is pressed.
- `drawing`: Boolean to track if the user is currently drawing a rectangle.
- `newPos`: Tuple storing the rectangle currently being drawn.
- `delete_mode`: Boolean indicating whether delete mode is active.
- `deleteList`: List of indices selected for deletion.
- `posList`: List of all saved rectangle positions.

### Behavior:

#### In **Delete Mode** (`delete_mode == True`):
- **Left Click** (`EVENT_LBUTTONDOWN`):
  - Toggles selection of a rectangle.
  - If a rectangle contains the clicked point:
    - Adds it to `deleteList` if not already selected.
    - Removes it from `deleteList` if it was already selected.

#### In **Draw Mode**:
- **Left Click Down** (`EVENT_LBUTTONDOWN`):
  - Starts drawing a rectangle and stores the starting point.
- **Mouse Move** (`EVENT_MOUSEMOVE`):
  - Updates `newPos` to show a preview of the rectangle being drawn.
- **Left Click Up** (`EVENT_LBUTTONUP`):
  - Finalizes the rectangle.
  - Appends the rectangle to `posList` and saves it to a file `Pos` using `pickle`.
- **Right Click** (`EVENT_RBUTTONDOWN`):
  - Deletes the rectangle under the cursor (if any) from `posList` and updates the `Pos` file.

> 💾 Rectangle positions are saved in a file named **`Pos`** after every addition or deletion in draw mode. 

In [90]:
def mouseClick(event, x, y, flags, param):
    global ix, iy, drawing, newPos, delete_mode, deleteList, posList

    if delete_mode:
        if event == cv2.EVENT_LBUTTONDOWN:
            for i, pos in enumerate(posList):
                x1, y1, w, h = pos
                if x1 < x < x1 + w and y1 < y < y1 + h:
                    if i not in deleteList:
                        deleteList.append(i)
                    else:
                        deleteList.remove(i)
    else:
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing = True
            ix, iy = x, y

        elif event == cv2.EVENT_MOUSEMOVE:
            if drawing:
                newPos = (ix, iy, x - ix, y - iy)

        elif event == cv2.EVENT_LBUTTONUP:
            drawing = False
            x1, y1 = min(ix, x), min(iy, y)
            w, h = abs(x - ix), abs(y - iy)
            posList.append((x1, y1, w, h))
            newPos = None
            with open('Pos', 'wb') as f:
                pickle.dump(posList, f)

        elif event == cv2.EVENT_RBUTTONDOWN:
            for i, pos in enumerate(posList):
                x1, y1, w, h = pos
                if x1 < x < x1 + w and y1 < y < y1 + h:
                    posList.pop(i)
                    with open('Pos', 'wb') as f:
                        pickle.dump(posList, f)
                    break

The bellow function allows users to annotate slots on an image using OpenCV. Users can draw rectangles or any symmentric shapes representing areas or switch to delete mode to remove them.

### Key Features:
- **Draw Mode**: Allows users to draw new rectangles by clicking and dragging.
- **Delete Mode**: Press `'d'` to toggle delete mode. In this mode:
  - Click rectangles to select them (they turn red).
  - Press `'x'` to delete all selected rectangles.
- **Rectangle Indexing**: Each rectangle is numbered for easy identification.
- **Visual Feedback**: 
  - Green rectangle = New position being drawn.
  - Pink rectangle = Normal  slot drawn.
  - Red rectangle = Selected for deletion.
- **Data Persistence**: Saves remaining positions to a binary file `kPos` using `pickle`.

### Controls:
- Press **`d`** to toggle draw/delete mode.
- In delete mode, press **`x`** to remove selected rectangles.
- Press **`q`** to exit the tool and close all OpenCV windows.

In [97]:
def draw():
    global delete_mode, deleteList, posList, newPos

    img = cv2.imread(img_1).copy()
    while True:
        img_display = img.copy()

        for i, (x1, y1, w, h) in enumerate(posList):
            color = (0, 0, 255) if i in deleteList else (255, 0, 255)
            cv2.rectangle(img_display, (x1, y1), (x1 + w, y1 + h), color, 2)
            cv2.putText(img_display, str(i + 1), (x1 + 5, y1 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        if newPos is not None:
            x1, y1, w, h = newPos
            cv2.rectangle(img_display, (x1, y1), (x1 + w, y1 + h), (0, 255, 0), 2)

        mode_text = "DELETE MODE [d] - Press [x] to delete selected" if delete_mode else "DRAW MODE [d]"
        cv2.putText(img_display, mode_text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)

        cv2.imshow("Image", img_display)
        cv2.setMouseCallback("Image", mouseClick)

        key = cv2.waitKey(1)
        if key == ord('d'):
            delete_mode = not delete_mode
            deleteList = []
        elif key == ord('x') and delete_mode:
            posList = [pos for i, pos in enumerate(posList) if i not in deleteList]
            with open('Pos', 'wb') as f:
                pickle.dump(posList, f)
            deleteList = []
            delete_mode = False
        elif key == ord('q'):
            break

    cv2.destroyAllWindows()

draw()


In [98]:
# Load video
cap = cv2.VideoCapture('Warehouseshelf.mp4')

# Reload positions if needed
with open('Pos', 'rb') as f:
    posList = pickle.load(f)

# Function to check space availability
def checkSpace(imgPro, imgDisplay):
    spaceCounter = 0

    for pos in posList:
     x, y, w, h = pos
     imgCrop = imgPro[y:y + h, x:x + w]
     count = cv2.countNonZero(imgCrop)

     if count < 300:
        color = (0, 255, 0)
        thickness = 5
        spaceCounter += 1
     else:
        color = (0, 0, 255)
        thickness = 2

     cv2.rectangle(imgDisplay, (x, y), (x + w, y + h), color, thickness)
     cvzone.putTextRect(imgDisplay, str(count), (x, y + h - 3), scale=1,
                       thickness=2, offset=0, colorR=color)

    cvzone.putTextRect(imgDisplay, f'Free: {spaceCounter}/{len(posList)}', (100, 50),
                       scale=3, thickness=5, offset=20, colorR=(0, 200, 0))

    return imgDisplay


In [99]:
def read_frame(cap):
    """Reads a frame from the video and handles loop reset when reaching the end."""
    if cap.get(cv2.CAP_PROP_POS_FRAMES) == cap.get(cv2.CAP_PROP_FRAME_COUNT):
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    success, img = cap.read()
    if not success:
        print("Failed to read frame.")
        return None
    return img

In [100]:
def preprocess_image(img):
    """Converts the image to grayscale, applies Gaussian blur, adaptive thresholding, median blur, and dilation."""
    imgGray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    imgBlur = cv2.GaussianBlur(imgGray, (3, 3), 1)
    imgThreshold = cv2.adaptiveThreshold(imgBlur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY_INV, 25, 16)
    imgMedian = cv2.medianBlur(imgThreshold, 5)
    kernel = np.ones((3, 3), np.uint8)
    imgDilate = cv2.dilate(imgMedian, kernel, iterations=1)
    return imgDilate

In [101]:
def display_images(imgGray, imgBlur, imgThreshold, imgMedian, imgDilate, imgResult):
    """Displays the different stages of image processing."""
    cv2.imshow("Gray", imgGray)
    cv2.imshow("Blur", imgBlur)
    cv2.imshow("Threshold", imgThreshold)
    cv2.imshow("Median", imgMedian)
    cv2.imshow("Dilated", imgDilate)
    cv2.imshow("Parking Detection", imgResult)

### In the bellow cell if user wants to see all the image processing tecniques 
- Just uncomment this bellow line
- display_images(imgDilate, imgDilate, imgDilate, imgDilate, imgDilate, imgResult)
- Break loop on 'q' key

In [102]:
def main():
    cap = cv2.VideoCapture("Warehouseshelf.mp4")  

    while True:
        img = read_frame(cap)
        if img is None:
            break
        
        imgDilate = preprocess_image(img)
        imgResult = checkSpace(imgDilate, img)
        
        
        #display_images(imgDilate, imgDilate, imgDilate, imgDilate, imgDilate, imgResult)
        cv2.imshow("Space Detection", imgResult)

        if cv2.waitKey(10) == ord('q'): # Break loop on 'q' key
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()